# Influence of homophily and heterophily on oversmoothing of GAT and TransformerConv

In [115]:
import torch
import torch_geometric
from typing import Any
import itertools
from torch_geometric.datasets import HeterophilousGraphDataset, CitationFull
import json
import copy
import os

In [116]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[INFO] device in use {DEVICE}")

[INFO] device in use cuda


## 1 Datasets

### 1.1 High homophily graphs

#### 1.1.1 pubmed dataset

In [117]:
path_pubmed = "./data/PubMed"

In [118]:
pubmed_dataset = torch_geometric.datasets.Planetoid(
    root=path_pubmed,
    name="PubMed",
    transform=torch_geometric.transforms.NormalizeFeatures()
)

In [119]:
print("Dataset information")
print("===================")
print(f"Number of graphs in dataset: {len(pubmed_dataset)}")
print(f"Number of features: {pubmed_dataset.num_features}")
print(f"Number of classes: {pubmed_dataset.num_classes}")

Dataset information
Number of graphs in dataset: 1
Number of features: 500
Number of classes: 3


In [120]:
pubmed_data = pubmed_dataset[0]
print(pubmed_data)

Data(x=[19717, 500], edge_index=[2, 88648], y=[19717], train_mask=[19717], val_mask=[19717], test_mask=[19717])


In [121]:
print("Graph information")
print("==================")
print(f"Number of nodes: {pubmed_data.num_nodes}")
print(f"Number of edges: {pubmed_data.num_edges}")
print(f"Has isolated nodes: {pubmed_data.has_isolated_nodes()}")
print(f"Has self loops: {pubmed_data.has_self_loops()}")

Graph information
Number of nodes: 19717
Number of edges: 88648
Has isolated nodes: False
Has self loops: False


In [122]:
h_score = torch_geometric.utils.homophily(pubmed_data.edge_index, pubmed_data.y, method="edge")
print(f"Edge Homophily Score: {h_score}")
h_score_adjusted = torch_geometric.utils.homophily(pubmed_data.edge_index, pubmed_data.y, method="edge_insensitive")
print(f"Edge insensitive homophily score: {h_score_adjusted}")

Edge Homophily Score: 0.8023869395256042
Edge insensitive homophily score: 0.6641403436660767


#### 1.1.2 CoraFull dataset

In [123]:
path_corafull = "./data/Corafull"

corafull_dataset = CitationFull(
    root=path_corafull,
    name="Cora",
    transform=torch_geometric.transforms.NormalizeFeatures()
)

In [124]:
print("Dataset information")
print("===================")
print(f"Number of graphs in dataset: {len(corafull_dataset)}")
print(f"Number of features: {corafull_dataset.num_features}")
print(f"Number of classes: {corafull_dataset.num_classes}")

Dataset information
Number of graphs in dataset: 1
Number of features: 8710
Number of classes: 70


In [125]:
corafull_data = corafull_dataset[0]
print(corafull_data)

Data(x=[19793, 8710], edge_index=[2, 126842], y=[19793])


In [126]:
print("Graph information")
print("==================")
print(f"Number of nodes: {corafull_data.num_nodes}")
print(f"Number of edges: {corafull_data.num_edges}")
print(f"Has isolated nodes: {corafull_data.has_isolated_nodes()}")
print(f"Has self loops: {corafull_data.has_self_loops()}")

Graph information
Number of nodes: 19793
Number of edges: 126842
Has isolated nodes: False
Has self loops: False


In [127]:
h_score = torch_geometric.utils.homophily(corafull_data.edge_index, corafull_data.y, method="edge")
print(f"Edge Homophily Score: {h_score}")
h_score_adjusted = torch_geometric.utils.homophily(corafull_data.edge_index, corafull_data.y, method="edge_insensitive")
print(f"Edge insensitive homophily score: {h_score_adjusted}")

Edge Homophily Score: 0.5670361518859863
Edge insensitive homophily score: 0.49586164951324463


### 1.2 Low homophily graphs

#### 1.2.1 roman empire

In [128]:
path_empire = "./data/Roman-empire"

dataset_empire = HeterophilousGraphDataset(root=path_empire, name="Roman-empire")

In [129]:
print("Dataset information")
print("=====================")
print(f"Number of graphs in dataset: {len(dataset_empire)}")
print(f"Number of features: {dataset_empire.num_features}")
print(f"Number of classes: {dataset_empire.num_classes}")

Dataset information
Number of graphs in dataset: 1
Number of features: 300
Number of classes: 18


In [130]:
data_empire = dataset_empire[0]
print(data_empire)

Data(x=[22662, 300], edge_index=[2, 65854], y=[22662], train_mask=[22662, 10], val_mask=[22662, 10], test_mask=[22662, 10])


In [131]:
print("Graph information")
print("==================")
print(f"Number of nodes: {data_empire.num_nodes}")
print(f"Number of edges: {data_empire.num_edges}")
print(f"Has isolated nodes: {data_empire.has_isolated_nodes()}")
print(f"Has self loops: {data_empire.has_self_loops()}")

Graph information
Number of nodes: 22662
Number of edges: 65854
Has isolated nodes: False
Has self loops: False


In [132]:
h_score = torch_geometric.utils.homophily(data_empire.edge_index, data_empire.y, method="edge")
print(f"Edge homophily score: {h_score}")
h_score_adjusted = torch_geometric.utils.homophily(data_empire.edge_index, data_empire.y, method="edge_insensitive")
print(f"Edge insensitive homophily score: {h_score_adjusted}")

Edge homophily score: 0.04689160734415054
Edge insensitive homophily score: 0.020823771134018898


#### 1.2.2 minesweeper dataset

In [133]:
path_minesweeper = "./data/Minesweeper"

dataset_minesweeper = HeterophilousGraphDataset(root=path_minesweeper, name="Minesweeper", transform=torch_geometric.transforms.NormalizeFeatures())

In [134]:
print("Dataset information")
print("=====================")
print(f"Number of graphs in dataset: {len(dataset_minesweeper)}")
print(f"Number of features: {dataset_minesweeper.num_features}")
print(f"Number of classes: {dataset_minesweeper.num_classes}")

Dataset information
Number of graphs in dataset: 1
Number of features: 7
Number of classes: 2


In [135]:
data_minesweeper = dataset_minesweeper[0]
print(data_minesweeper)

Data(x=[10000, 7], edge_index=[2, 78804], y=[10000], train_mask=[10000, 10], val_mask=[10000, 10], test_mask=[10000, 10])


In [136]:
print("Graph information")
print("==================")
print(f"Number of nodes: {data_minesweeper.num_nodes}")
print(f"Number of edges: {data_minesweeper.num_edges}")
print(f"Has isolated nodes: {data_minesweeper.has_isolated_nodes()}")
print(f"Has self loops: {data_minesweeper.has_self_loops()}")

Graph information
Number of nodes: 10000
Number of edges: 78804
Has isolated nodes: False
Has self loops: False


In [137]:
h_score = torch_geometric.utils.homophily(data_minesweeper.edge_index, data_minesweeper.y, method="edge")
print(f"Edge homophily score: {h_score}")
h_score_adjusted = torch_geometric.utils.homophily(data_minesweeper.edge_index, data_minesweeper.y, method="edge_insensitive")
print(f"Edge insensitive homophily score: {h_score_adjusted}")

Edge homophily score: 0.6827825903892517
Edge insensitive homophily score: 0.009364798665046692


## 2 GAT network definition

In [138]:
class GATBlock(torch.nn.Module):
    def __init__(self, in_channels, hidden_size, heads, dropout_rate):
        super().__init__()
        self.conv = torch_geometric.nn.GATv2Conv(
            in_channels, 
            hidden_size, 
            heads=heads, 
            concat=True, 
            dropout=dropout_rate
        )
        
        self.project_back = torch.nn.Linear(hidden_size * heads, in_channels)
        self.dropout = torch.nn.Dropout(dropout_rate)

    def forward(self, x, edge_index):
        x = self.conv(x, edge_index)
        x = self.project_back(x)
        return self.dropout(x)

class GATNetwork(torch.nn.Module):
    def __init__(self, number_of_layers, input_size, hidden_size, output_size, num_heads, dropout_rate):
        super().__init__()
        self.layers = torch.nn.ModuleList()
        self.norms = torch.nn.ModuleList()

        self.input_proj = torch.nn.Linear(input_size, hidden_size)

        for _ in range(number_of_layers):
            self.norms.append(torch_geometric.nn.LayerNorm(hidden_size))
            self.layers.append(GATBlock(hidden_size, hidden_size, num_heads, dropout_rate))

        self.post_lin = torch.nn.Linear(hidden_size, output_size)

    def forward(self, x, edge_index):
        x = self.input_proj(x)
        
        for norm, layer in zip(self.norms, self.layers):
            identity = x 
            
            x = norm(x)
            x = layer(x, edge_index)
            x = torch.nn.functional.gelu(x)

            x = x + identity
                
        return self.post_lin(x)

## 3 TransformerConv network definition

In [139]:
class TransformerBlock(torch.nn.Module):
    def __init__(self, in_channels, hidden_size, heads, dropout_rate):
        super().__init__()
        self.conv = torch_geometric.nn.conv.TransformerConv(in_channels, hidden_size, heads=heads, concat=True, dropout=dropout_rate)
        
        self.project_back = torch.nn.Linear(hidden_size * heads, in_channels)
        self.dropout = torch.nn.Dropout(dropout_rate)

    def forward(self, x, edge_index):
        x = self.conv(x, edge_index)
        x = self.project_back(x) 
        return self.dropout(x)

class TransformerConvNetwork(torch.nn.Module):
    def __init__(self, number_of_layers, input_size, hidden_size, output_size, num_heads, dropout_rate):
        super().__init__()
        self.layers = torch.nn.ModuleList()
        self.norms = torch.nn.ModuleList()
        
        self.input_proj = torch.nn.Linear(input_size, hidden_size)

        for _ in range(number_of_layers):
            self.norms.append(torch_geometric.nn.LayerNorm(hidden_size))
            self.layers.append(TransformerBlock(hidden_size, hidden_size, num_heads, dropout_rate))

        self.post_lin = torch.nn.Linear(hidden_size, output_size)

    def forward(self, x, edge_index):
        x = self.input_proj(x)
        
        for norm, layer in zip(self.norms, self.layers):
            identity = x 
            
            x = norm(x)
            x = layer(x, edge_index)
            x = torch.nn.functional.gelu(x)
            
            x = x + identity
                
        return self.post_lin(x)

# 4 Transductive learning setting

In [140]:
class TransductiveTrainer:
    def __init__(
        self,
        model: torch.nn.Module,
        optimizer: torch.optim.Optimizer,
        loss_function: torch.nn.Module,
        data: torch_geometric.data.Data,
        epochs: int
    ):
        self.model = model
        self.optimizer = optimizer
        self.loss_function = loss_function
        self.data = data
        self.epochs = epochs
        self.logger = {
            "train_loss": [],
            "train_accuracy": [],
            "val_loss": [],
            "val_accuracy": []
        }
    
    def standardize_splits(self, data):
        for mask_name in ['train_mask', 'val_mask', 'test_mask']:
            if hasattr(data, mask_name):
                mask = getattr(data, mask_name)
                if mask.dim() > 1:
                    setattr(data, mask_name, mask[:, 0])
                if getattr(data, mask_name).dtype != torch.bool:
                    new_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
                    new_mask[mask] = True
                    setattr(data, mask_name, new_mask)
        return data

    def fit(self, patiance: int = 5):

        self.data = self.standardize_splits(self.data)

        best_val_acc = 0.0
        epochs_passed = 0

        for epoch in range(self.epochs):
            # training step
            self.model.train()
            logits = self.model(self.data.x, self.data.edge_index)
            train_loss_value = self.loss_function(logits[self.data.train_mask], self.data.y[self.data.train_mask])
            self.optimizer.zero_grad()
            train_loss_value.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
            self.optimizer.step()
            train_accuracy = self.accuracy(logits[self.data.train_mask].argmax(dim=1), self.data.y[self.data.train_mask])

            train_loss_value = train_loss_value.detach().cpu().numpy()
            train_accuracy = train_accuracy.detach().cpu().numpy()
            print(f"[INFO] epoch {epoch + 1}:\n\ttraining loss: {train_loss_value}")
            print(f"\n\ttraining accuracy: {train_accuracy}")
            self.logger["train_loss"].append(train_loss_value)
            self.logger["train_accuracy"].append(train_accuracy)

            # validation step
            self.model.eval()
            with torch.no_grad():
                logits = self.model(self.data.x, self.data.edge_index)
                val_loss_value = self.loss_function(logits[self.data.val_mask], self.data.y[self.data.val_mask])
                val_accuracy = self.accuracy(logits[self.data.val_mask].argmax(dim=1), self.data.y[self.data.val_mask])
                val_loss_value = val_loss_value.cpu().numpy()
                val_accuracy = float(val_accuracy.cpu().numpy())

                print(f"\n\tval loss: {val_loss_value}")
                print(f"\n\tval accuracy: {val_accuracy}")
                print(f"\n")
                self.logger["val_loss"].append(val_loss_value)
                self.logger["val_accuracy"].append(val_accuracy)

                # early stopping
                if val_accuracy > best_val_acc:
                    best_model_state = copy.deepcopy(self.model.state_dict())
                    best_val_acc = val_accuracy
                    epochs_passed = 0
                else:
                    epochs_passed += 1
                    if epochs_passed >= patiance:
                        break
        
        self.model.load_state_dict(best_model_state)
    
    def accuracy(self, y_predict: torch.Tensor, y_truth: torch.Tensor):
        return torch.sum(y_predict == y_truth) / len(y_truth)

    def compute_dirichlet_energy(self, x: torch.Tensor, edge_index: torch.Tensor):
        row, col = edge_index
        # Normalize node features to unit hypersphere to ensure energy 
        # is about angle between nodes and not just raw magnitude.
        x_norm = torch.nn.functional.normalize(x, p=2, dim=-1)
        
        source, target = x_norm[row], x_norm[col]
        squared_diff = torch.pow(source - target, 2).sum(dim=-1)
        
        return float(squared_diff.mean().cpu().numpy())

    def compute_gdr(self, x: torch.Tensor):
        # 1. Normalize embeddings to remove magnitude bias
        x = torch.nn.functional.normalize(x, p=2, dim=-1)
        y = self.data.y
        num_classes = y.max().item() + 1
        
        centroids = []
        intra_distances = []

        for i in range(num_classes):
            mask = (y == i)
            if mask.sum() == 0: continue
            
            # 2. Calculate Centroid for class i
            class_nodes = x[mask]
            centroid = class_nodes.mean(dim=0)
            centroids.append(centroid)
            
            # 3. Calculate Intra-class distance (Mean distance to centroid)
            # Using Euclidean distance here, Cosine distance is also an option
            dist = torch.norm(class_nodes - centroid, p=2, dim=1).mean()
            intra_distances.append(dist)

        # 4. Average Intra-class distance across all classes
        d_intra = torch.stack(intra_distances).mean()

        # 5. Calculate Inter-class distance (Average distance between centroids)
        centroids = torch.stack(centroids)
        # Pairwise distance between class centroids
        inter_dist_matrix = torch.cdist(centroids, centroids, p=2)
        
        # We only want the upper triangle (excluding diagonal 0s)
        num_centroids = centroids.size(0)
        triu_indices = torch.triu_indices(num_centroids, num_centroids, offset=1)
        d_inter = inter_dist_matrix[triu_indices[0], triu_indices[1]].mean()

        # 6. Final Ratio
        return (d_inter / (d_intra + 1e-8)).item()

    def compute_mad_gap(self, x: torch.Tensor):
        # 1. Normalize node features to the unit hypersphere
        # This ensures we measure 'direction' (Oversmoothing) rather than 'magnitude'
        x_norm = torch.nn.functional.normalize(x, p=2, dim=-1)
        n = x_norm.size(0)

        # 2. Calculate Global MAD (Average distance between random nodes)
        # We sample up to 2000 nodes to keep the NxN matrix calculation efficient
        sample_size = min(n, 2000)
        idx = torch.randperm(n)[:sample_size]
        x_sample = x_norm[idx]
        
        # Cosine Similarity Matrix: [sample_size, sample_size]
        sim_global = torch.mm(x_sample, x_sample.t())
        
        # Convert to Distance: Dist = 1 - Similarity
        # Excluding the diagonal (self-similarity) for a true average
        mad_global = (1 - sim_global).sum() / (sample_size * (sample_size - 1))

        # 3. Calculate Target MAD (Average distance between neighbors)
        row, col = self.data.edge_index
        # Efficient dot product for normalized vectors to get cosine similarity
        neighbor_sim = (x_norm[row] * x_norm[col]).sum(dim=-1)
        mad_tgt = (1 - neighbor_sim).mean()

        # 4. The MAD-Gap
        # If this is high, neighbors are distinct from the rest of the graph.
        # If this is zero, neighbors are indistinguishable from random nodes.
        return (mad_global - mad_tgt).item()
    
    @torch.no_grad()
    def get_oversmoothing_metrics(self):
        self.model.eval()
        logits = self.model(self.data.x, self.data.edge_index)
        energy = self.compute_dirichlet_energy(logits, self.data.edge_index)
        gdr = self.compute_gdr(logits)
        mad_gap =self.compute_mad_gap(logits)

        return energy, gdr, mad_gap
    
    @torch.no_grad()
    def test(self, y_predict: torch.Tensor | None = None, y_truth: torch.Tensor | None = None):
        self.model.eval()
        if y_predict is None and y_truth is None:
            logits = self.model(self.data.x, self.data.edge_index)
            y_predict = logits[self.data.test_mask].argmax(dim=1)
            y_truth = self.data.y[self.data.test_mask]
        test_accuracy = self.accuracy(y_predict, y_truth)

        return float(test_accuracy.cpu().numpy())

# 5 Experiment class

In [141]:
class Experiment:
    def __init__(
        self,
        layers: list[int],
        datasets: list[torch_geometric.data.Dataset],
        epochs: int,
        base_layers: int,
        hyperparameter_grid: dict[str, list[Any]],
        hyperparams_optim_epochs: int,
        hyperparameters_path: str,
        save_log_path: str
    ):
        self._hyperparameter_grid = hyperparameter_grid
        self._layers = layers
        self._datasets = datasets
        self._base_layers = base_layers
        self._epochs = epochs
        self._hyperparams_optim_epochs = hyperparams_optim_epochs
        self._hyperparameters_path = hyperparameters_path
        self._save_log_path = save_log_path

        self._best_hyperparams_gat = []
        self._best_hyperparams_transformer = []
        self._best_base_accs_gat = []
        self._best_base_accs_trans = []

        self._log = {}
    
    def save_log(self):
        with open(self._save_log_path, "w") as file:
            json.dump(self._log, file)
    
    def perform_experiment(self):
        #1. find best hyperparameters on each dataset for both architectures using smaller models
        # (heads, hidden_dim_per_head) are paired in such a way that hidden dimensionality is always the same
        if os.path.exists(self._hyperparameters_path):
            print("[INFO] Hyperparameters found!")
            with open(self._hyperparameters_path, "r") as file:
                hyperparameters_data = json.load(file)
                self._best_hyperparams_gat = hyperparameters_data["gat"]
                self._best_hyperparams_transformer = hyperparameters_data["trans"]
        else:
            print("[INFO] hyperparameters are being optimized!")
            self.optimize_hyperparameters()

        #2. for each dataset, load best hyperparameters for both architectures
        # iterate over provided layers and train both architectures for each layer count
        for dataset_idx, dataset in enumerate(self._datasets):
            metrics = {
                "gat": {
                    "acc": [],
                    "energy": [],
                    "mad_gap": [],
                    "gdr": []
                },
                "trans": {
                    "acc": [],
                    "energy": [],
                    "mad_gap": [],
                    "gdr": []
                }
            }
            for layer_num in self._layers:
                heads_gat, hidden_dim_gat = self._best_hyperparams_gat[dataset_idx]["heads_hidden_dim"]
                gat_trainer = self.get_trainers(
                    number_of_layers = layer_num,
                    input_size = dataset.num_features,
                    hidden_dim = hidden_dim_gat,
                    output_size = dataset.num_classes,
                    heads = heads_gat,
                    dropout = self._best_hyperparams_gat[dataset_idx]["dropout"],
                    lr = self._best_hyperparams_gat[dataset_idx]["lr"],
                    weight_decay = self._best_hyperparams_gat[dataset_idx]["weight_decay"],
                    data = dataset[0],
                    epochs = self._epochs,
                    model_type = "gat"
                )[0]

                heads_trans, hidden_dim_trans = self._best_hyperparams_transformer[dataset_idx]["heads_hidden_dim"]
                trans_trainer = self.get_trainers(
                    number_of_layers = layer_num,
                    input_size = dataset.num_features,
                    hidden_dim = hidden_dim_trans,
                    output_size = dataset.num_classes,
                    heads = heads_trans,
                    dropout = self._best_hyperparams_transformer[dataset_idx]["dropout"],
                    lr = self._best_hyperparams_transformer[dataset_idx]["lr"],
                    weight_decay = self._best_hyperparams_transformer[dataset_idx]["weight_decay"],
                    data = dataset[0],
                    epochs = self._epochs,
                    model_type = "trans"
                )[0]

                gat_trainer.fit()
                trans_trainer.fit()

                accuracy = gat_trainer.test()
                energy, gdr, mad_gap = gat_trainer.get_oversmoothing_metrics()
                metrics["gat"]["acc"].append(accuracy)
                metrics["gat"]["energy"].append(energy)
                metrics["gat"]["gdr"].append(gdr)
                metrics["gat"]["mad_gap"].append(mad_gap)

                accuracy = trans_trainer.test()
                energy, gdr, mad_gap = trans_trainer.get_oversmoothing_metrics()
                metrics["trans"]["acc"].append(accuracy)
                metrics["trans"]["energy"].append(energy)
                metrics["trans"]["gdr"].append(gdr)
                metrics["trans"]["mad_gap"].append(mad_gap)
            
            self._log[dataset.name] = metrics

        self.save_log()

        return self._log

    def optimize_hyperparameters(self):
        for dataset in self._datasets:
            best_gat, best_trans, best_gat_accuracy, best_trans_accuracy = self.grid_search(dataset)
            self._best_hyperparams_gat.append(best_gat)
            self._best_hyperparams_transformer.append(best_trans)
            self._best_base_accs_gat.append(best_gat_accuracy)
            self._best_base_accs_trans.append(best_trans_accuracy)
        
        hyperparams_data = {
            "datasets": [dataset.name for dataset in self._datasets],
            "gat": self._best_hyperparams_gat,
            "trans": self._best_hyperparams_transformer
        }

        with open(self._hyperparameters_path, "w") as file:
            json.dump(hyperparams_data, file)
    
    def get_combination_generator(self):
        keys = self._hyperparameter_grid.keys()
        values = self._hyperparameter_grid.values()
        hyperparams_combinations = itertools.product(*values)
        for combination in hyperparams_combinations:
            yield dict(zip(keys, combination))
    
    def get_trainers(
        self,
        number_of_layers: int,
        input_size: int,
        hidden_dim: int,
        output_size: int,
        heads: int,
        dropout: float,
        lr: float,
        weight_decay: float,
        data: torch_geometric.data.Data,
        epochs: int,
        model_type: str | None = None
    ):
        trainers = []

        if model_type == "gat" or model_type == None:
            gat_base = GATNetwork(
                number_of_layers = number_of_layers,
                input_size = input_size,
                hidden_size = hidden_dim,
                output_size = output_size,
                num_heads = heads,
                dropout_rate = dropout
            ).to(DEVICE)
            optimizer_gat = torch.optim.Adam(
                params = gat_base.parameters(),
                lr = lr,
                weight_decay = weight_decay
            )
            gat_trainer = TransductiveTrainer(
                model = gat_base,
                optimizer = optimizer_gat,
                loss_function = torch.nn.CrossEntropyLoss(),
                data = data,
                epochs = epochs
            )
            trainers.append(gat_trainer)

        if model_type == "trans" or model_type == None:
            trans_base = TransformerConvNetwork(
                number_of_layers = number_of_layers,
                input_size = input_size,
                hidden_size = hidden_dim,
                output_size = output_size,
                num_heads = heads,
                dropout_rate = dropout
            ).to(DEVICE)
            optimizer_trans = torch.optim.Adam(
                params = trans_base.parameters(),
                lr = lr,
                weight_decay = weight_decay
            )
            trans_trainer = TransductiveTrainer(
                model = trans_base,
                optimizer = optimizer_trans,
                loss_function = torch.nn.CrossEntropyLoss(),
                data = data,
                epochs = epochs
            )
            trainers.append(trans_trainer)

        return trainers   

    def grid_search(
        self,
        dataset: torch_geometric.data.Dataset
    ):
        hyperparameter_generator = self.get_combination_generator()

        best_gat_accuracy = 0.0
        best_trans_accuracy = 0.0
        best_gat = None
        best_trans = None

        for hyperparameters in hyperparameter_generator:
            heads, hidden_dim = hyperparameters["heads_hidden_dim"]
            
            gat_trainer, trans_trainer = self.get_trainers(
                number_of_layers = self._base_layers,
                input_size = dataset.num_features,
                hidden_dim = hidden_dim,
                output_size = dataset.num_classes,
                heads = heads,
                dropout = hyperparameters["dropout"],
                lr = hyperparameters["lr"],
                weight_decay = hyperparameters["weight_decay"],
                data = dataset[0],
                epochs = self._hyperparams_optim_epochs
            )

            gat_trainer.fit()
            trans_trainer.fit()
            gat_accuracy = gat_trainer.test()
            trans_accuracy = trans_trainer.test()
            if gat_accuracy > best_gat_accuracy:
                best_gat_accuracy = gat_accuracy
                best_gat = hyperparameters
            if trans_accuracy > best_trans_accuracy:
                best_trans_accuracy = trans_accuracy
                best_trans = hyperparameters
        
        return best_gat, best_trans, best_gat_accuracy, best_trans_accuracy

In [113]:
experiment = Experiment(
    layers = [2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 26, 28, 30, 32, 34, 36, 38, 40, 42, 46, 48],
    datasets = [pubmed_dataset.to(DEVICE), corafull_dataset.to(DEVICE), dataset_empire.to(DEVICE), data_minesweeper.to(DEVICE)],
    epochs = 200,
    base_layers = 10,
    hyperparameter_grid = {
        "heads_hidden_dim": [(2, 64), (4, 32), (8, 16)],
        "dropout": [0.1, 0.2, 0.3, 0.4, 0.5],
        "lr": [1e-2, 5e-2, 1e-3, 5e-3, 1e-4, 5e-4],
        "weight_decay": [5e-6, 5e-5, 5e-4, 5e-3]
    },
    hyperparams_optim_epochs = 30,
    hyperparameters_path="./best_hyperparameters.json",
    save_log_path="./data_log.json"
)

In [142]:
experiment = Experiment(
    layers = [2, 4],
    datasets = [pubmed_dataset.to(DEVICE), corafull_dataset.to(DEVICE), dataset_empire.to(DEVICE), data_minesweeper.to(DEVICE)],
    epochs = 2,
    base_layers = 2,
    hyperparameter_grid = {
        "heads_hidden_dim": [(2, 64), (4, 32)],
        "dropout": [0.1, 0.2],
        "lr": [1e-2, 5e-2],
        "weight_decay": [5e-6, 5e-5]
    },
    hyperparams_optim_epochs = 2,
    hyperparameters_path="./best_hyperparameters.json",
    save_log_path="./data_log.json"
)

In [ ]:
log = experiment.perform_experiment()
print(log)

[INFO] hyperparameters are being optimized!
[INFO] epoch 1:
	training loss: 1.0982738733291626

	training accuracy: 0.2666666805744171

	val loss: 1.8036683797836304

	val accuracy: 0.3880000114440918


[INFO] epoch 2:
	training loss: 1.6104309558868408

	training accuracy: 0.36666667461395264

	val loss: 1.7620240449905396

	val accuracy: 0.19600000977516174


[INFO] epoch 1:
	training loss: 1.0912091732025146

	training accuracy: 0.3333333432674408

	val loss: 1.8169047832489014

	val accuracy: 0.41600000858306885


[INFO] epoch 2:
	training loss: 1.7456376552581787

	training accuracy: 0.3333333432674408

	val loss: 2.7109880447387695

	val accuracy: 0.19600000977516174


[INFO] epoch 1:
	training loss: 1.1006568670272827

	training accuracy: 0.31666669249534607

	val loss: 2.117568254470825

	val accuracy: 0.19600000977516174


[INFO] epoch 2:
	training loss: 1.638260006904602

	training accuracy: 0.3500000238418579

	val loss: 1.6901377439498901

	val accuracy: 0.3880000114440918


AttributeError: 'GlobalStorage' object has no attribute 'train_mask'